# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library and the Croissant schema.

### Dataset Source
The dataset schema (Croissant) is available at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

We'll use this in the following steps.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant
# (Optional) Install seaborn for nice plots later
!pip install seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(url)

# Display the dataset's metadata: name and description
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'Unknown')}")

## 2. Data Overview
Review available record sets, their IDs, fields, and columns. All are referenced via their `@id`.

### Listing Record Sets
We list each record set, its `@id`, and the associated fields and columns.

*(If your dataset is large, you may wish to limit the printout)*

In [ ]:
# List record sets, fields, and columns in the metadata by @id.
print("Record sets found in the dataset:\n")
record_sets = dataset.metadata.recordSet
for i, rs in enumerate(record_sets):
    print(f"{i+1}. RecordSet name  : {getattr(rs, 'name', '')}")
    print(f"   RecordSet @id   : {rs.id}")
    # List fields for this record set
    if hasattr(rs, 'field'):
        print("   Fields in this RecordSet:")
        for f in rs.field:
            print(f"      - {getattr(f, 'name', '')} (@id: {f.id})")
    # List columns (data source columns, typically CSV)
    if hasattr(rs, 'column'):
        print("   Columns in this RecordSet:")
        for c in rs.column:
            print(f"      - {getattr(c, 'name', '')} (@id: {c.id})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Pick your desired record set and field IDs from the overview. Here, we use the first available record set for demonstration.

In [ ]:
# Get list of all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.recordSet]
print("Available RecordSet @id's:", record_set_ids)

# Choose one record set for demonstration (first one)
main_record_set_id = record_set_ids[0]
print("\nLoading records from RecordSet @id:", main_record_set_id)

# Load all records from the selected RecordSet
records = list(dataset.records(record_set=main_record_set_id))
# Convert to DataFrame
df = pd.DataFrame(records)
print("\nDataFrame columns (fields by @id):\n", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data inspection and basic processing: filtering, normalization, and grouping.

Below we will:
- Inspect summary stats
- Filter on a numeric field
- Normalize that field
- Optionally group by a categorical field (if available)

**Update the field IDs below as appropriate for your chosen record set.**

In [ ]:
# Let's find a numeric field (example: molecular weight, so we search for a field like 'mw' or containing 'weight')
print("All columns:", df.columns.tolist())
# Try to infer a numeric field column
# For this dataset, common proteomics fields are: 'cr:mw', 'cr:pi', 'cr:abundance_sample1', etc.

import re
numeric_cols = [c for c in df.columns if re.search(r'(mw|weight|abundance|peptide|coverage|count|number|ratio|pi)', c, re.IGNORECASE)]
print("Possible numeric fields:", numeric_cols)

# Assign a numeric field for further processing
if numeric_cols:
    numeric_field = numeric_cols[0]  # Pick first likely candidate
else:
    raise Exception("No numeric field found!")

# Filter: e.g., keep only proteins/rows where the selected field > threshold
threshold = df[numeric_field].dropna().mean()  # Use mean as a threshold for demonstration
filtered_df = df[df[numeric_field] > threshold]

print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

# Normalize the field (z-score normalization)
field_mean = filtered_df[numeric_field].mean()
field_std = filtered_df[numeric_field].std()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - field_mean) / field_std

print(f"\nNormalized values for {numeric_field} (first 5 rows):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try a group by, e.g., by protein 'accession' or 'cr:accession', if it exists
group_fields = [c for c in df.columns if re.search(r'(accession|description|sample|type|group|category)', c, re.IGNORECASE)]
group_field = group_fields[0] if group_fields else None

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped mean {numeric_field} by {group_field} (first 5 groups):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships. Here are demonstration plots for the main numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histograms for the numeric field before and after normalization
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.histplot(df[numeric_field].dropna(), kde=True, color='skyblue')
plt.title(f"Histogram of {numeric_field}")

plt.subplot(1,2,2)
sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, color='orange')
plt.title(f"Histogram: Normalized {numeric_field} (Filtered)")
plt.tight_layout()
plt.show()

# Optional: Boxplot by group if available
if group_field:
    plt.figure(figsize=(10,4))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
In this notebook, we explored the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the Croissant standard and `mlcroissant` library:
* Loaded the rich dataset schema and records
* Identified record sets, fields, and columns (referenced by `@id`)
* Analyzed and visualized quantitative aspects like protein abundance, molecular weight, or peptide counts

With ease of loading, referencing, and interacting with fields by `@id`, `mlcroissant` supports reproducible, standards-based dataset use for downstream science and machine learning workflows.

You can now extend this analysis as needed: try exploring other record sets or fields identified above!